# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose **Random Forest** because this lane is a binary classification problem and the data contains a mix of numeric and categorical signals with potentially non-linear relationships.

Random Forest is a good baseline learned model because it can capture interactions between signals without requiring a linear relationship. It also gives feature importance, which makes the model easier to inspect.

I will compare it with my Week-4 hand-written baseline using the **same data, same metric, and same client-holdout split**.

In [3]:
# =========================================================
# ML-08 — SETUP
# =========================================================

from pathlib import Path
import sys
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

# Load prepared feature vector
DATA_PATH = REPO_ROOT / "data" / "processed" / "refresh_feature_vector.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Prepared feature vector not found: {DATA_PATH}"
    )

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (30000, 52)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'has_clicks', 'has_ai_sessions', 'measurable_opportunity']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a **grouped client-holdout split** rather than randomly splitting individual rows.

The reason is that multiple rows can belong to the same client. A random row split could therefore put pages from the same client into both training and test data, making the evaluation too optimistic.

Holding out complete clients gives a more honest estimate of how the model behaves on clients it did not see during training.

The split is therefore approximately 80% train clients and 20% test clients.

In [4]:
# =========================================================
# 2. CLIENT-HOLDOUT SPLIT
# =========================================================

TARGET = "is_declining_label"
GROUP = "client_id"

if TARGET not in df.columns:
    raise KeyError(
        f"Missing target column: {TARGET}"
    )

if GROUP not in df.columns:
    raise KeyError(
        f"Missing grouping column: {GROUP}"
    )

# Check target
print("Target distribution:")
print(df[TARGET].value_counts(dropna=False))
print()

# Remove rows without a target
model_df = df.dropna(subset=[TARGET]).copy()

# Make target integer
model_df[TARGET] = model_df[TARGET].astype(int)

# Grouped split by client
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        model_df[TARGET],
        groups=model_df[GROUP]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("TRAIN")
print("Rows:", len(train_df))
print("Clients:", train_df[GROUP].nunique())

print("\nTEST")
print("Rows:", len(test_df))
print("Clients:", test_df[GROUP].nunique())

# Confirm no client appears in both sets
train_clients = set(train_df[GROUP])
test_clients = set(test_df[GROUP])

overlap = train_clients.intersection(test_clients)

print("\nClient overlap:", len(overlap))

assert len(overlap) == 0, "Client leakage detected!"

print("\nTarget rate:")
print(
    pd.DataFrame({
        "train_rate": [train_df[TARGET].mean()],
        "test_rate": [test_df[TARGET].mean()]
    })
)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

TRAIN
Rows: 23837
Clients: 25

TEST
Rows: 6163
Clients: 7

Client overlap: 0

Target rate:
   train_rate  test_rate
0    0.550111   0.510952


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I train the Random Forest using only information available before the outcome.

I exclude identifiers and leakage-prone fields, especially `trend_direction`, `trend_pct`, and the target itself.

The evaluation uses the same binary target and client-holdout idea as the baseline comparison.

The main metric is **Precision@50**, because the practical question is which 50 pages should be reviewed first.

In [6]:
# =========================================================
# 3. TRAIN RANDOM FOREST + EVALUATE
# =========================================================

# ---------------------------------------------------------
# Feature selection
# ---------------------------------------------------------

DROP_COLUMNS = {
    # Target
    "is_declining_label",

    # Direct label/leakage fields
    "trend_direction",
    "trend_pct",

    # Identifiers
    "content_id",
    "client_id",
}

feature_cols = [
    c for c in model_df.columns
    if c not in DROP_COLUMNS
]

X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[TARGET].copy()

print("Candidate features:", len(feature_cols))

# ---------------------------------------------------------
# Separate numeric / categorical columns
# ---------------------------------------------------------

numeric_features = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

# ---------------------------------------------------------
# Preprocessing
# ---------------------------------------------------------

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop"
)

# ---------------------------------------------------------
# Random Forest
# ---------------------------------------------------------

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model = Pipeline([
    ("preprocess", preprocessor),
    ("model", rf)
])

# ---------------------------------------------------------
# Train
# ---------------------------------------------------------

print("Training Random Forest...")

model.fit(X_train, y_train)

print("Training complete.")

# ---------------------------------------------------------
# Test probabilities
# ---------------------------------------------------------

test_probability = model.predict_proba(X_test)[:, 1]

test_predictions = (
    test_probability >= 0.5
).astype(int)

# ---------------------------------------------------------
# Standard classification metrics
# ---------------------------------------------------------

print("\nClassification metrics")

print(
    "Precision:",
    round(
        precision_score(
            y_test,
            test_predictions,
            zero_division=0
        ),
        4
    )
)

print(
    "Recall:",
    round(
        recall_score(
            y_test,
            test_predictions,
            zero_division=0
        ),
        4
    )
)

print(
    "F1:",
    round(
        f1_score(
            y_test,
            test_predictions,
            zero_division=0
        ),
        4
    )
)

# ---------------------------------------------------------
# Precision@50
# ---------------------------------------------------------

evaluation = test_df[
    ["content_id", "client_id", TARGET]
].copy()

evaluation["model_score"] = test_probability

evaluation = evaluation.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

top50 = evaluation.head(50)

precision_at_50 = top50[TARGET].mean()

print("\nPrecision@50:", round(precision_at_50, 4))
print(
    "Positive pages in top 50:",
    int(top50[TARGET].sum()),
    "/ 50"
)

# ---------------------------------------------------------
# Baseline comparison
# ---------------------------------------------------------

baseline_path = (
    REPO_ROOT
    / "data"
    / "processed"
    / "baseline_refresh_queue.csv"
)

if baseline_path.exists():

    baseline = pd.read_csv(baseline_path)

    print("\nBaseline columns:")
    print(baseline.columns.tolist())

    # Find a usable baseline score column
    possible_score_cols = [
        c for c in baseline.columns
        if c.lower() in [
            "score",
            "baseline_score",
            "action_score"
        ]
    ]

    if possible_score_cols:

        baseline_score_col = possible_score_cols[0]

        baseline_eval = baseline.merge(
            test_df[
                ["content_id", TARGET]
            ],
            on="content_id",
            how="inner"
        )

        baseline_eval = baseline_eval.sort_values(
            baseline_score_col,
            ascending=False
        )

        baseline_top50 = baseline_eval.head(50)

        baseline_p50 = (
            baseline_top50[TARGET].mean()
        )

        print(
            "Baseline Precision@50:",
            round(baseline_p50, 4)
        )

        print(
            "Model Precision@50:",
            round(precision_at_50, 4)
        )

        print(
            "Lift:",
            round(
                precision_at_50 - baseline_p50,
                4
            )
        )

        comparison = pd.DataFrame({
            "method": [
                "Week-4 baseline",
                "Random Forest"
            ],
            "precision_at_50": [
                baseline_p50,
                precision_at_50
            ]
        })

        display(comparison)

    else:
        print(
            "\nNo obvious baseline score column found."
        )
        print(
            "Use the printed baseline columns to "
            "identify the score column."
        )


Candidate features: 47
Numeric features: 36
Categorical features: 11
Training Random Forest...
Training complete.

Classification metrics
Precision: 0.6884
Recall: 0.7606
F1: 0.7227

Precision@50: 0.92
Positive pages in top 50: 46 / 50


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I will inspect the highest-scored false positives and the missed positive examples rather than relying only on the aggregate metric.

A false positive means the model ranked a page highly even though the measured label was negative.

A false negative means a positive page received a relatively low model score.

The model is decision-support: high score means "review first", not that the model has established a causal reason for the page's outcome.

In [7]:
# =========================================================
# 4. ERROR ANALYSIS
# =========================================================

# Add predictions
error_df = test_df[
    [
        "content_id",
        "client_id",
        TARGET
    ]
].copy()

error_df["model_score"] = test_probability

error_df["predicted"] = (
    error_df["model_score"] >= 0.5
).astype(int)

# ---------------------------------------------------------
# False positives
# ---------------------------------------------------------

false_positives = error_df[
    (error_df["predicted"] == 1)
    & (error_df[TARGET] == 0)
].sort_values(
    "model_score",
    ascending=False
)

print("Top false positives:")
display(
    false_positives.head(10)
)

# ---------------------------------------------------------
# False negatives
# ---------------------------------------------------------

false_negatives = error_df[
    (error_df["predicted"] == 0)
    & (error_df[TARGET] == 1)
].sort_values(
    "model_score",
    ascending=False
)

print("\nPositive pages missed by the 0.50 threshold:")
display(
    false_negatives.head(10)
)

# ---------------------------------------------------------
# Top 50 review
# ---------------------------------------------------------

print("\nTop 10 model-ranked pages:")
display(
    error_df
    .sort_values("model_score", ascending=False)
    .head(10)
)

# ---------------------------------------------------------
# Simple interpretation
# ---------------------------------------------------------

print("\nInterpretation:")
print(
    "The model should be treated as a ranking and review aid."
)
print(
    "High-scored false positives are cases where the "
    "model's ranking signal did not correspond to the label."
)
print(
    "Missed positives show where the model may under-rank "
    "pages that eventually meet the measured target."
)

Top false positives:


,content_id,client_id,is_declining_label,model_score,predicted
12332,content_4d9f36001f06,client_8527a891e2,0,0.846146,1
20736,content_41baf0722ad9,client_8527a891e2,0,0.844554,1
2357,content_8f1409b2674e,client_8527a891e2,0,0.839588,1
11061,content_0b47dae0c7f9,client_8527a891e2,0,0.814272,1
28718,content_ef6e7d7cfe15,client_8527a891e2,0,0.807458,1
5477,content_3164f3076003,client_8527a891e2,0,0.806866,1
4050,content_500bd3907331,client_4e07408562,0,0.804765,1
1517,content_816d77e36e14,client_8527a891e2,0,0.803534,1
10080,content_35d63627bf3e,client_8527a891e2,0,0.794836,1
29456,content_b46c62b14582,client_8527a891e2,0,0.793506,1



Positive pages missed by the 0.50 threshold:


,content_id,client_id,is_declining_label,model_score,predicted
19295,content_b855d7e44f8e,client_8527a891e2,1,0.499784,0
8223,content_9e722da54bd3,client_f369cb89fc,1,0.499456,0
11625,content_d7827558d408,client_4e07408562,1,0.499052,0
25714,content_a44f5dd132bb,client_8527a891e2,1,0.498857,0
12566,content_d9f0773a5c22,client_4e07408562,1,0.498411,0
7930,content_e55da9707bf5,client_4e07408562,1,0.497591,0
28116,content_accceffe127e,client_4e07408562,1,0.497569,0
9307,content_0a5e0981dc87,client_e629fa6598,1,0.497048,0
27043,content_de3344df3971,client_e629fa6598,1,0.496746,0
4954,content_15cb6f57162b,client_4e07408562,1,0.496536,0



Top 10 model-ranked pages:


,content_id,client_id,is_declining_label,model_score,predicted
3211,content_f55fd2d8ed04,client_4e07408562,1,0.887864,1
25063,content_29884c0f9255,client_8527a891e2,1,0.870950,1
14343,content_9ac61c04930e,client_8527a891e2,1,0.856651,1
4076,content_66458ac1b739,client_8527a891e2,1,0.854185,1
22429,content_3395aba722a2,client_8527a891e2,1,0.850760,1
8279,content_4fc70e470460,client_4e07408562,1,0.848584,1
2934,content_24d8b73697f6,client_4e07408562,1,0.848370,1
16859,content_fbbacf50108d,client_8527a891e2,1,0.846958,1
10019,content_c694763d4685,client_8527a891e2,1,0.846711,1
11764,content_866a6f5de5e2,client_4e07408562,1,0.846491,1



Interpretation:
The model should be treated as a ranking and review aid.
High-scored false positives are cases where the model's ranking signal did not correspond to the label.
Missed positives show where the model may under-rank pages that eventually meet the measured target.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.